# FlightAI — ticket prices from SQLite

Week 2 Day 4: Gradio chat assistant that looks up return-ticket prices via an OpenAI **tool call**, then reads from a local SQLite `prices` table.

Needs `OPENAI_API_KEY` in `.env`. Running the last cell creates `prices.db` in the current working directory.

In [ ]:
import json
import os
import sqlite3

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4.1-mini"

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

# JSON Schema the model sees. It cannot run Python; it can only *request*
# this function with a destination_city argument when the user asks for a price.
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}
tools = [{"type": "function", "function": price_function}]

In [ ]:
# SQLite file on disk. Context manager commits on success and always closes the connection.
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # PRIMARY KEY on city so we can upsert later without duplicate rows.
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )
    conn.commit()


def set_ticket_price(city, price):
    """Seed / update a city's price. Not exposed as an LLM tool in this notebook."""
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price),
        )
        conn.commit()


# Sample catalog so lookups succeed before anyone writes to the DB by hand.
for city, price in {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999,
    "hyderabad": 499,
}.items():
    set_ticket_price(city, price)


def get_ticket_price(city):
    """Tool implementation: parameterized SELECT so city is never concatenated into SQL."""
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        result = cursor.fetchone()  # None if the city is missing
        return (
            f"Ticket price to {city} is ${result[0]}"
            if result
            else "No price data available for this city"
        )


def handle_tool_calls(message):
    """Turn each model tool_call into a `role: tool` message the API requires."""
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            # Arguments arrive as a JSON string, not a Python dict.
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append(
                {
                    "role": "tool",
                    "content": price_details,
                    # tool_call_id ties this result back to the model's request.
                    "tool_call_id": tool_call.id,
                }
            )
    return responses


def chat(message, history):
    # Gradio `type="messages"` already uses role/content dicts; keep only those fields.
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [
        {"role": "system", "content": system_message}
    ] + history + [{"role": "user", "content": message}]

    # First call: model may answer in text *or* ask to run get_ticket_price.
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_results = handle_tool_calls(assistant_msg)
        # Replay the assistant's tool request, then our SQLite results, then ask again.
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type="messages").launch()